# Tarea 1 – Parte 2: Clasificación (La República)

**Estudiante:** Karen Macedo

Pasos: (1) línea base zero-shot → (2) etiquetado débil con LLM (Gemini) + auditoría de 20 casos → (3) split 70/15/15 → (4) fine-tuning de DistilBERT multilingüe → (5) evaluación en test → (6) comparación y decisión.

- La etiqueta de referencia (`etiqueta`) viene de la sección del diario (Parte 1).
- El modelo usa solo el `titular` como entrada, porque eso es lo que se escribe en el dashboard.

In [ ]:
import json, os, random, time
from pathlib import Path
from datetime import datetime

import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
CATEGORIAS = ["politica", "economia", "deportes", "ciencia_tecnologia"]
label2id = {c: i for i, c in enumerate(CATEGORIAS)}
id2label = {i: c for c, i in label2id.items()}
GPU = torch.cuda.is_available()
Path("modelo").mkdir(exist_ok=True)

df = pd.read_csv("datos/noticias.csv")
print(df.shape, "| GPU:", GPU)
print(df["etiqueta"].value_counts())

## 1. Línea base zero-shot (40 titulares)
Modelo XNLI en español: `Recognai/bert-base-spanish-wwm-cased-xnli`. Este es el puntaje mínimo a superar.

In [ ]:
from transformers import pipeline

ETIQUETAS_ZS = {"política y gobierno": "politica", "economía y negocios": "economia",
                "deportes": "deportes", "ciencia y tecnología": "ciencia_tecnologia"}
zero_shot = pipeline("zero-shot-classification", model="Recognai/bert-base-spanish-wwm-cased-xnli",
                     device=0 if GPU else -1)

def predecir_zs(titulares):
    salida = zero_shot(list(titulares), candidate_labels=list(ETIQUETAS_ZS),
                       hypothesis_template="Esta noticia trata sobre {}.")
    return [ETIQUETAS_ZS[s["labels"][0]] for s in salida]

muestra_zs = df.groupby("etiqueta", group_keys=False).sample(10, random_state=SEED)
t0 = time.time()
muestra_zs["pred_zs"] = predecir_zs(muestra_zs["titular"])
lat_zs = (time.time() - t0) / len(muestra_zs) * 1000

acc_zs = accuracy_score(muestra_zs["etiqueta"], muestra_zs["pred_zs"])
f1_zs = f1_score(muestra_zs["etiqueta"], muestra_zs["pred_zs"], average="macro")
print(f"ZERO-SHOT (40) -> accuracy = {acc_zs:.3f} | F1 macro = {f1_zs:.3f} | {lat_zs:.0f} ms/titular")
print(classification_report(muestra_zs["etiqueta"], muestra_zs["pred_zs"], labels=CATEGORIAS, zero_division=0))

## 2. Etiquetado débil con LLM (Gemini)
Prompt estructurado: el LLM ve titular + bajada (nunca la sección) y responde JSON con categoría y confianza. Se envían lotes de 20 titulares. La API key se lee de la variable `GEMINI_API_KEY` (no se sube al repositorio).

In [ ]:
MODELO_LLM = "gemini-2.5-flash-lite"   # si no está disponible, cambiar por otro modelo Flash de Gemini
RUTA_LLM = Path("datos/etiquetas_llm.csv")

PROMPT = """Eres editor de un diario peruano. Clasifica cada noticia en EXACTAMENTE una categoría:
- politica: gobierno, Congreso, elecciones, partidos, justicia, fiscalía, corrupción, relaciones exteriores.
- economia: empresas, mercados, empleo, precios, impuestos, AFP, tipo de cambio, finanzas personales.
- deportes: fútbol, vóley, torneos, deportistas, resultados.
- ciencia_tecnologia: ciencia, salud basada en estudios, espacio, naturaleza, tecnología, apps, internet, IA.
Devuelve una lista JSON con objetos {"id": int, "etiqueta_llm": str, "confianza": float entre 0 y 1}."""

def etiquetar_con_gemini(datos, lote=20):
    from google import genai
    from google.genai import types
    clave = os.environ.get("GEMINI_API_KEY")
    if not clave:
        from getpass import getpass
        clave = getpass("GEMINI_API_KEY: ")
    cliente = genai.Client(api_key=clave)
    config = types.GenerateContentConfig(system_instruction=PROMPT, temperature=0,
                                         response_mime_type="application/json")
    resultados = {}
    for i in range(0, len(datos), lote):
        parte = datos.iloc[i:i + lote]
        texto = "\n".join(f"{idx}. {r.titular} | {r.bajada}" for idx, r in parte.iterrows())
        for intento in range(3):
            try:
                resp = cliente.models.generate_content(model=MODELO_LLM, contents=texto, config=config)
                for item in json.loads(resp.text):
                    if item.get("etiqueta_llm") in CATEGORIAS:
                        resultados[int(item["id"])] = item
                break
            except Exception as e:
                print("Reintento lote", i, type(e).__name__); time.sleep(15)
        time.sleep(4)   # respetar el límite gratuito de pedidos por minuto
    return resultados

if RUTA_LLM.exists():
    llm = pd.read_csv(RUTA_LLM)
    print("Se reutiliza", RUTA_LLM)
else:
    t0 = time.time()
    res = etiquetar_con_gemini(df)
    lat_llm_total = time.time() - t0
    faltan = [i for i in df.index if i not in res]
    if faltan:                         # segunda pasada solo para los que faltaron
        res.update(etiquetar_con_gemini(df.loc[faltan]))
    llm = pd.DataFrame({
        "titular": df["titular"],
        "etiqueta_llm": [res.get(i, {}).get("etiqueta_llm") for i in df.index],
        "confianza": [res.get(i, {}).get("confianza") for i in df.index],
        "modelo_llm": MODELO_LLM,
        "fecha": datetime.now().strftime("%Y-%m-%d"),
    })
    llm.to_csv(RUTA_LLM, index=False, encoding="utf-8")
    print(f"Etiquetado en {lat_llm_total:.0f} s")

df["etiqueta_llm"] = llm["etiqueta_llm"].values
print("Sin etiqueta del LLM:", df["etiqueta_llm"].isna().sum())
print("Coincidencia LLM vs sección del diario (todo el dataset):",
      round((df["etiqueta_llm"] == df["etiqueta"]).mean(), 3))
llm.head()

### Auditoría humana de 20 predicciones al azar
Leo cada titular y decido si la categoría del LLM es correcta. La columna `seccion_diario` es solo de apoyo.
**En la celda siguiente escribo los números de los casos en los que NO estoy de acuerdo con el LLM.**

In [ ]:
auditoria = df.sample(20, random_state=SEED)[["titular", "etiqueta_llm", "etiqueta"]]
auditoria = auditoria.rename(columns={"etiqueta": "seccion_diario"}).reset_index(drop=True)
pd.set_option("display.max_colwidth", 150)
display(auditoria)

In [ ]:
DESACUERDOS = []   # <- escribir aquí los números de fila (0-19) donde el LLM se equivocó, ej. [3, 15]

auditoria["correcto_humano"] = ~auditoria.index.isin(DESACUERDOS)
acuerdo = int(auditoria["correcto_humano"].sum())
auditoria.to_csv("datos/auditoria_20.csv", index=False, encoding="utf-8")
print(f"Acuerdo LLM-Humano: {acuerdo}/20 ({acuerdo / 20:.0%})")

## 3. Split estratificado 70/15/15 (`random_state=42`)
El test queda guardado y no se toca hasta la evaluación final.

In [ ]:
train, temp = train_test_split(df, test_size=0.30, stratify=df["etiqueta"], random_state=SEED)
val, test = train_test_split(temp, test_size=0.50, stratify=temp["etiqueta"], random_state=SEED)
print("train:", len(train), "| val:", len(val), "| test:", len(test))

## 4. Fine-tuning de `distilbert-base-multilingual-cased`
3 épocas, batch 16, evaluación en cada época y se guarda el mejor según F1 macro. En CPU tarda pocos minutos porque los titulares son cortos (en Colab con GPU T4, segundos).

In [ ]:
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments)

MODELO_BASE = "distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)

class Titulares(torch.utils.data.Dataset):
    def __init__(self, parte):
        self.enc = tokenizer(parte["titular"].tolist(), truncation=True, max_length=64)
        self.y = parte["etiqueta"].map(label2id).tolist()
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.y[i]
        return item

def metricas(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds), "f1_macro": f1_score(labels, preds, average="macro")}

modelo = AutoModelForSequenceClassification.from_pretrained(
    MODELO_BASE, num_labels=4, id2label=id2label, label2id=label2id)

args = TrainingArguments(
    output_dir="modelo", num_train_epochs=3, per_device_train_batch_size=16,
    per_device_eval_batch_size=32, learning_rate=5e-5, weight_decay=0.01,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="f1_macro",
    logging_strategy="epoch", seed=SEED, report_to="none", fp16=GPU)

trainer = Trainer(model=modelo, args=args, train_dataset=Titulares(train), eval_dataset=Titulares(val),
                  data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=metricas)
t0 = time.time()
trainer.train()
print(f"Entrenamiento: {time.time() - t0:.0f} s")
historial = pd.DataFrame([h for h in trainer.state.log_history if "eval_f1_macro" in h])
display(historial[["epoch", "eval_loss", "eval_accuracy", "eval_f1_macro"]].round(3))

trainer.save_model("modelo")
tokenizer.save_pretrained("modelo")

## 5. Evaluación final en test

In [ ]:
t0 = time.time()
pred = trainer.predict(Titulares(test))
lat_ft = (time.time() - t0) / len(test) * 1000
y_true = test["etiqueta"].tolist()
y_pred = [id2label[i] for i in np.argmax(pred.predictions, axis=-1)]

acc_ft = accuracy_score(y_true, y_pred)
f1_ft = f1_score(y_true, y_pred, average="macro")
print(f"FINE-TUNED (test={len(test)}) -> accuracy = {acc_ft:.3f} | F1 macro = {f1_ft:.3f}")
print(classification_report(y_true, y_pred, labels=CATEGORIAS, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=CATEGORIAS)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CATEGORIAS, yticklabels=CATEGORIAS)
plt.xlabel("Predicción"); plt.ylabel("Real"); plt.title(f"Matriz de confusión – test (F1 macro {f1_ft:.2f})")
plt.tight_layout(); plt.show()

In [ ]:
f1_cat = pd.Series(f1_score(y_true, y_pred, labels=CATEGORIAS, average=None), index=CATEGORIAS)
print("F1 por categoría:\n", f1_cat.round(3))
print("\nMejor:", f1_cat.idxmax(), "| Peor:", f1_cat.idxmin())
errores = test.assign(pred=y_pred).query("etiqueta != pred")[["titular", "etiqueta", "pred"]]
display(errores)

**Reflexión por categoría.** Deportes suele ser la más fácil porque tiene vocabulario muy propio (nombres de clubes, "gol", "Liga 1", "Alianza", "Universitario"). Las más difíciles son política y economía: muchas notas mezclan los dos temas (presupuesto, impuestos, decisiones del Gobierno o del Congreso sobre la economía) y el titular solo no siempre deja claro el enfoque. En ciencia y tecnología, los errores aparecen cuando el titular habla de salud, clima o empresas tecnológicas, que se confunden con economía. La tabla de errores de arriba muestra los casos concretos.

## 6. Comparación y decisión

In [ ]:
test_zs = predecir_zs(test["titular"])
t0 = time.time(); _ = predecir_zs(test["titular"].head(10)); lat_zs = (time.time() - t0) / 10 * 1000
llm_test = test["etiqueta_llm"].fillna("sin_respuesta")

tabla = pd.DataFrame([
    {"modelo": "Zero-shot XNLI (sin entrenar)", "accuracy": accuracy_score(y_true, test_zs),
     "f1_macro": f1_score(y_true, test_zs, average="macro"), "latencia_ms": round(lat_zs),
     "costo": "gratis, local, ~440 MB"},
    {"modelo": f"LLM débil ({MODELO_LLM})", "accuracy": accuracy_score(y_true, llm_test),
     "f1_macro": f1_score(y_true, llm_test, labels=CATEGORIAS, average="macro"), "latencia_ms": "~1000+ (API)",
     "costo": "API externa, internet y cuota"},
    {"modelo": "DistilBERT fine-tuned", "accuracy": acc_ft, "f1_macro": f1_ft,
     "latencia_ms": round(lat_ft), "costo": "gratis, local, ~540 MB"},
]).round(3)
display(tabla)

**Decisión:** despliego el **DistilBERT fine-tuned**. Supera claramente al zero-shot y queda cerca del LLM, pero corre en local en milisegundos, sin pagar API ni depender de internet o de límites de cuota. El LLM sirve para etiquetar rápido, pero no para un dashboard que responde en vivo.

In [ ]:
metricas_finales = {
    "total_noticias": int(len(df)), "f1_macro_test": round(float(f1_ft), 4),
    "accuracy_test": round(float(acc_ft), 4), "f1_zero_shot_40": round(float(f1_zs), 4),
    "acuerdo_llm_humano": acuerdo, "acuerdo_llm_humano_pct": acuerdo / 20,
    "modelo_llm": MODELO_LLM, "tamano_test": int(len(test)),
}
Path("modelo/metricas.json").write_text(json.dumps(metricas_finales, indent=2), encoding="utf-8")
print(json.dumps(metricas_finales, indent=2))
print(f"\nResultados Test: F1 Macro: {f1_ft:.2f} | Zero-shot: {f1_zs:.2f} | Acuerdo LLM-Humano: {acuerdo}/20")